# Course 16: Build & Deploy ML Solutions on Vertex AI

**GCP Professional Machine Learning Engineer — Exam Sections 1, 3, 4, 6**

This notebook provides hands-on practice with:
- AutoML tabular: create dataset, train, evaluate (SDK, commented for cost)
- Custom training: define a training script, create custom job config
- Deploy model to endpoint (SDK, commented for cost)
- Batch prediction setup
- Model monitoring configuration
- Endpoint traffic splitting pattern
- Cost estimation for different deployment configs

---

## Setup & Installations

In [ ]:
# Install required packages
!pip install -q google-cloud-aiplatform pandas scikit-learn

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Configuration — update these for your GCP project
PROJECT_ID = "your-gcp-project-id"  # CHANGE THIS
REGION = "us-central1"
BUCKET_URI = f"gs://{PROJECT_ID}-vertex-staging"

print(f"Project: {PROJECT_ID}")
print(f"Region: {REGION}")
print(f"Bucket: {BUCKET_URI}")
print("\nUpdate PROJECT_ID above before running GCP-dependent cells.")

---
## 1. AutoML Tabular: Create Dataset, Train, Evaluate

AutoML handles architecture search, hyperparameter tuning, and feature engineering.
You provide data and an objective — Vertex AI does the rest.

> **Cost Warning**: AutoML training incurs charges. All GCP API calls are commented by default.

In [ ]:
# Create a synthetic dataset to understand the data format
import pandas as pd
import numpy as np

np.random.seed(42)
n_samples = 5000

df = pd.DataFrame({
    "transaction_amount": np.random.exponential(100, n_samples).round(2),
    "merchant_category": np.random.choice(
        ["grocery", "electronics", "gas", "restaurant", "online"], n_samples
    ),
    "hour_of_day": np.random.randint(0, 24, n_samples),
    "day_of_week": np.random.randint(0, 7, n_samples),
    "distance_from_home": np.random.exponential(10, n_samples).round(1),
    "is_international": np.random.choice([0, 1], n_samples, p=[0.9, 0.1]),
    "num_transactions_last_hour": np.random.poisson(2, n_samples),
})

# Synthetic label: fraud is more likely for high amounts, late night, international
fraud_score = (
    (df["transaction_amount"] > 300).astype(float) * 0.3 +
    (df["hour_of_day"].isin([23, 0, 1, 2, 3])).astype(float) * 0.2 +
    df["is_international"] * 0.3 +
    (df["num_transactions_last_hour"] > 5).astype(float) * 0.2
)
df["is_fraud"] = (fraud_score + np.random.normal(0, 0.15, n_samples) > 0.5).astype(int)

print(f"Dataset shape: {df.shape}")
print(f"Fraud rate: {df['is_fraud'].mean():.2%}")
print(f"\nSample rows:")
df.head()

In [ ]:
# Save dataset locally (would upload to GCS or BigQuery for AutoML)
df.to_csv("/tmp/fraud_data.csv", index=False)
print("Dataset saved to /tmp/fraud_data.csv")
print(f"File size: {pd.io.common.file_exists('/tmp/fraud_data.csv')}")
print(f"\nColumn types for AutoML column_transformations:")
for col in df.columns:
    dtype = "categorical" if df[col].dtype == "object" else "numeric"
    print(f"  {col}: {dtype}")

In [ ]:
# ============================================================
# AUTOML TRAINING (commented to avoid costs)
# ============================================================

# from google.cloud import aiplatform
#
# aiplatform.init(project=PROJECT_ID, location=REGION, staging_bucket=BUCKET_URI)
#
# # Step 1: Create managed dataset
# dataset = aiplatform.TabularDataset.create(
#     display_name="fraud-detection-dataset",
#     gcs_source="gs://your-bucket/fraud_data.csv",
#     # OR from BigQuery:
#     # bq_source="bq://project.dataset.table",
# )
# print(f"Dataset created: {dataset.resource_name}")
#
# # Step 2: Configure AutoML training job
# job = aiplatform.AutoMLTabularTrainingJob(
#     display_name="fraud-automl-classifier",
#     optimization_prediction_type="classification",
#     optimization_objective="maximize-au-prc",  # best for imbalanced data
#     column_transformations=[
#         {"numeric": {"column_name": "transaction_amount"}},
#         {"categorical": {"column_name": "merchant_category"}},
#         {"numeric": {"column_name": "hour_of_day"}},
#         {"numeric": {"column_name": "day_of_week"}},
#         {"numeric": {"column_name": "distance_from_home"}},
#         {"numeric": {"column_name": "is_international"}},
#         {"numeric": {"column_name": "num_transactions_last_hour"}},
#     ],
# )
#
# # Step 3: Run training
# model = job.run(
#     dataset=dataset,
#     target_column="is_fraud",
#     budget_milli_node_hours=2000,  # 2 node-hours
#     training_fraction_split=0.8,
#     validation_fraction_split=0.1,
#     test_fraction_split=0.1,
#     model_display_name="fraud-automl-v1",
# )
# print(f"Model trained: {model.resource_name}")
#
# # Step 4: Evaluate
# model_eval = model.list_model_evaluations()[0]
# print(f"AUC-PR: {model_eval.metrics.get('auPrc', 'N/A')}")
# print(f"AUC-ROC: {model_eval.metrics.get('auRoc', 'N/A')}")
# print(f"Log Loss: {model_eval.metrics.get('logLoss', 'N/A')}")

print("AutoML training code ready (commented to avoid costs).")
print("Key parameters:")
print("  optimization_objective='maximize-au-prc'  # for imbalanced classification")
print("  budget_milli_node_hours=2000              # 2 node-hours of search")
print("  training/validation/test split: 80/10/10")

---
## 2. Custom Training: Define Script & Job Config

For full control, write your own training script and submit it as a custom training job.
Vertex AI provides pre-built containers for TensorFlow, PyTorch, scikit-learn, and XGBoost.

In [ ]:
# Write a training script that Vertex AI will execute
import os

TRAINER_DIR = "/tmp/vertex_trainer"
os.makedirs(TRAINER_DIR, exist_ok=True)

training_script = '''
"""Custom training script for Vertex AI.

This script:
1. Loads data from GCS (via AIP_TRAINING_DATA_URI env var)
2. Trains a GradientBoosting model
3. Evaluates and logs metrics
4. Saves model to AIP_MODEL_DIR for deployment
"""
import argparse
import os
import pickle
import json

import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score
)


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--n_estimators", type=int, default=100)
    parser.add_argument("--learning_rate", type=float, default=0.1)
    parser.add_argument("--max_depth", type=int, default=5)
    parser.add_argument("--data_path", type=str,
                        default=os.environ.get("AIP_TRAINING_DATA_URI", "/tmp/fraud_data.csv"))
    parser.add_argument("--model_dir", type=str,
                        default=os.environ.get("AIP_MODEL_DIR", "/tmp/model"))
    return parser.parse_args()


def main():
    args = parse_args()
    print(f"Training with: n_estimators={args.n_estimators}, "
          f"lr={args.learning_rate}, max_depth={args.max_depth}")

    # Load data
    df = pd.read_csv(args.data_path)
    print(f"Loaded {len(df)} samples")

    # Prepare features (one-hot encode categoricals)
    df_encoded = pd.get_dummies(df, columns=["merchant_category"])
    X = df_encoded.drop(columns=["is_fraud"])
    y = df_encoded["is_fraud"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # Train
    model = GradientBoostingClassifier(
        n_estimators=args.n_estimators,
        learning_rate=args.learning_rate,
        max_depth=args.max_depth,
        random_state=42,
    )
    model.fit(X_train, y_train)

    # Evaluate
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "auc_roc": roc_auc_score(y_test, y_prob),
        "auc_pr": average_precision_score(y_test, y_prob),
    }

    print("\n=== Evaluation Metrics ===")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")

    # Save model
    os.makedirs(args.model_dir, exist_ok=True)
    model_path = os.path.join(args.model_dir, "model.pkl")
    with open(model_path, "wb") as f:
        pickle.dump(model, f)

    # Save feature names (needed for serving)
    features_path = os.path.join(args.model_dir, "features.json")
    with open(features_path, "w") as f:
        json.dump({"features": list(X.columns)}, f)

    print(f"\nModel saved to {model_path}")
    print(f"Features saved to {features_path}")


if __name__ == "__main__":
    main()
'''

script_path = os.path.join(TRAINER_DIR, "task.py")
with open(script_path, "w") as f:
    f.write(training_script)

print(f"Training script written to {script_path}")

In [ ]:
# Run the training script locally to verify it works
!python /tmp/vertex_trainer/task.py \
    --data_path /tmp/fraud_data.csv \
    --model_dir /tmp/local_model \
    --n_estimators 50 \
    --learning_rate 0.1 \
    --max_depth 4

In [ ]:
# Verify the saved model works for prediction
import pickle
import json

with open("/tmp/local_model/model.pkl", "rb") as f:
    loaded_model = pickle.load(f)

with open("/tmp/local_model/features.json") as f:
    feature_info = json.load(f)

print(f"Model type: {type(loaded_model).__name__}")
print(f"Number of features: {len(feature_info['features'])}")
print(f"Features: {feature_info['features'][:5]}...")

# Test prediction
sample = pd.get_dummies(df.drop(columns=["is_fraud"]), columns=["merchant_category"]).iloc[:3]
predictions = loaded_model.predict_proba(sample)[:, 1]
print(f"\nSample predictions (fraud probability): {predictions.round(4)}")

In [ ]:
# ============================================================
# SUBMIT CUSTOM TRAINING JOB TO VERTEX AI (commented for cost)
# ============================================================

# from google.cloud import aiplatform
#
# aiplatform.init(project=PROJECT_ID, location=REGION, staging_bucket=BUCKET_URI)
#
# # Option A: Pre-built container (simplest)
# job = aiplatform.CustomTrainingJob(
#     display_name="fraud-custom-sklearn",
#     script_path="/tmp/vertex_trainer/task.py",
#     container_uri="us-docker.pkg.dev/vertex-ai/training/scikit-learn-cpu.1-3:latest",
#     requirements=["pandas"],
#     model_serving_container_image_uri=(
#         "us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-3:latest"
#     ),
# )
#
# model = job.run(
#     replica_count=1,
#     machine_type="n1-standard-4",
#     args=[
#         "--n_estimators", "200",
#         "--learning_rate", "0.05",
#         "--max_depth", "6",
#         "--data_path", f"gs://{PROJECT_ID}-data/fraud_data.csv",
#     ],
# )
# print(f"Model: {model.resource_name}")
#
# # Option B: With GPU (for deep learning)
# gpu_job = aiplatform.CustomTrainingJob(
#     display_name="fraud-custom-pytorch",
#     script_path="trainer/train_pytorch.py",
#     container_uri="us-docker.pkg.dev/vertex-ai/training/pytorch-gpu.2-1:latest",
#     requirements=["transformers"],
# )
# model = gpu_job.run(
#     replica_count=1,
#     machine_type="n1-standard-8",
#     accelerator_type="NVIDIA_TESLA_T4",
#     accelerator_count=1,
# )

print("Custom training job configurations ready (commented for cost).")
print("\nPre-built container URIs:")
print("  sklearn: us-docker.pkg.dev/vertex-ai/training/scikit-learn-cpu.1-3:latest")
print("  tf-gpu:  us-docker.pkg.dev/vertex-ai/training/tf-gpu.2-14:latest")
print("  pytorch: us-docker.pkg.dev/vertex-ai/training/pytorch-gpu.2-1:latest")
print("  xgboost: us-docker.pkg.dev/vertex-ai/training/xgboost-cpu.1-7:latest")

---
## 3. Deploy Model to Endpoint

Once trained, deploy the model to an online endpoint for real-time predictions.
Vertex AI handles load balancing, auto-scaling, and health checks.

> **Cost Warning**: Running endpoints incurs per-hour charges. Code is commented.

In [ ]:
# ============================================================
# DEPLOY MODEL TO ENDPOINT (commented for cost)
# ============================================================

# from google.cloud import aiplatform
#
# aiplatform.init(project=PROJECT_ID, location=REGION)
#
# # Reference the trained model
# model = aiplatform.Model("projects/PROJECT/locations/REGION/models/MODEL_ID")
#
# # Option A: Create endpoint and deploy in one step
# endpoint = model.deploy(
#     deployed_model_display_name="fraud-model-v1",
#     machine_type="n1-standard-4",
#     min_replica_count=1,
#     max_replica_count=5,
#     traffic_percentage=100,
#     # For GPU:
#     # accelerator_type="NVIDIA_TESLA_T4",
#     # accelerator_count=1,
# )
#
# # Option B: Create endpoint first, then deploy (more control)
# endpoint = aiplatform.Endpoint.create(
#     display_name="fraud-detection-endpoint",
#     description="Production fraud detection endpoint",
# )
# model.deploy(
#     endpoint=endpoint,
#     deployed_model_display_name="fraud-model-v1",
#     machine_type="n1-standard-4",
#     min_replica_count=1,
#     max_replica_count=5,
# )
#
# # Make predictions
# response = endpoint.predict(
#     instances=[
#         {"transaction_amount": 500.0, "merchant_category": "electronics",
#          "hour_of_day": 2, "day_of_week": 5, "distance_from_home": 50.0,
#          "is_international": 1, "num_transactions_last_hour": 8},
#     ]
# )
# print(f"Predictions: {response.predictions}")

print("Endpoint deployment code ready (commented for cost).")
print("\nKey deployment parameters:")
print("  machine_type: n1-standard-4 (or with GPU for DL models)")
print("  min_replica_count: 1 (0 = scale to zero, cold start ~30-60s)")
print("  max_replica_count: 5 (auto-scales based on traffic)")
print("  traffic_percentage: 100 (for single model; split for A/B testing)")

---
## 4. Batch Prediction Setup

For large-scale offline scoring, batch prediction is more cost-effective than online endpoints.
Vertex AI auto-provisions compute, runs predictions, and writes results to GCS or BigQuery.

In [ ]:
# ============================================================
# BATCH PREDICTION (commented for cost)
# ============================================================

# from google.cloud import aiplatform
#
# aiplatform.init(project=PROJECT_ID, location=REGION)
# model = aiplatform.Model("projects/PROJECT/locations/REGION/models/MODEL_ID")
#
# # Option A: From GCS (JSONL format)
# batch_job = model.batch_predict(
#     job_display_name="fraud-batch-scoring-jan",
#     gcs_source="gs://my-bucket/batch_input/transactions.jsonl",
#     gcs_destination_prefix="gs://my-bucket/batch_output/",
#     instances_format="jsonl",
#     predictions_format="jsonl",
#     machine_type="n1-standard-4",
#     starting_replica_count=2,
#     max_replica_count=10,
# )
#
# # Option B: From BigQuery to BigQuery
# batch_job = model.batch_predict(
#     job_display_name="fraud-batch-scoring-bq",
#     bigquery_source=f"bq://{PROJECT_ID}.ml_data.new_transactions",
#     bigquery_destination_prefix=f"bq://{PROJECT_ID}.ml_data",
#     instances_format="bigquery",
#     predictions_format="bigquery",
#     machine_type="n1-standard-4",
#     starting_replica_count=2,
#     max_replica_count=10,
# )
#
# # Monitor progress
# batch_job.wait()
# print(f"Batch job state: {batch_job.state}")
# print(f"Output: {batch_job.output_info}")

print("Batch prediction setup ready (commented for cost).")
print("\nSupported formats:")
print("  Input:  JSONL (GCS), CSV (GCS), TFRecord (GCS), BigQuery")
print("  Output: JSONL (GCS), BigQuery")
print("\nCost advantage: no always-on endpoint; compute provisioned only during job.")

In [ ]:
# Prepare a sample JSONL file for batch prediction
import json

# Create sample batch input
batch_instances = df.drop(columns=["is_fraud"]).head(10).to_dict(orient="records")

jsonl_path = "/tmp/batch_input.jsonl"
with open(jsonl_path, "w") as f:
    for instance in batch_instances:
        f.write(json.dumps(instance) + "\n")

print(f"Sample batch input ({len(batch_instances)} instances):")
for i, inst in enumerate(batch_instances[:3]):
    print(f"  Instance {i}: {json.dumps(inst)[:80]}...")
print(f"\nSaved to {jsonl_path}")
print("Upload to GCS: gsutil cp /tmp/batch_input.jsonl gs://your-bucket/batch_input/")

---
## 5. Model Monitoring Configuration

Model monitoring detects when input features or predictions drift from their training-time
distributions. This is critical for catching model degradation before it impacts users.

In [ ]:
# ============================================================
# MODEL MONITORING SETUP (commented for cost)
# ============================================================

# from google.cloud import aiplatform
#
# aiplatform.init(project=PROJECT_ID, location=REGION)
#
# # Reference existing endpoint
# endpoint = aiplatform.Endpoint("projects/PROJECT/locations/REGION/endpoints/ENDPOINT_ID")
#
# # Configure monitoring job
# monitoring_job = aiplatform.ModelDeploymentMonitoringJob.create(
#     display_name="fraud-model-monitoring",
#     endpoint=endpoint,
#
#     # How much prediction traffic to sample
#     logging_sampling_strategy={
#         "random_sample_config": {"sample_rate": 0.8}  # log 80% of requests
#     },
#
#     # How often to run analysis
#     monitoring_frequency=1,  # every 1 hour
#
#     # Skew detection: training data vs serving data
#     monitoring_config={
#         "skew_detection_config": {
#             "data_source": f"bq://{PROJECT_ID}.ml_data.training_features",
#             "skew_thresholds": {
#                 "transaction_amount": {"value": 0.3},
#                 "merchant_category": {"value": 0.3},
#                 "hour_of_day": {"value": 0.3},
#                 "distance_from_home": {"value": 0.3},
#             },
#         },
#         # Drift detection: recent predictions vs baseline predictions
#         "drift_detection_config": {
#             "drift_thresholds": {
#                 "transaction_amount": {"value": 0.3},
#                 "merchant_category": {"value": 0.3},
#                 "hour_of_day": {"value": 0.3},
#             },
#         },
#         # Attribution drift: feature importance changes
#         "explanation_config": {
#             "enable_feature_attributes": True,
#         },
#     },
#
#     # Alert configuration
#     alert_config={
#         "email_alert_config": {
#             "user_emails": ["ml-team@company.com"]
#         },
#         # Can also send to Cloud Monitoring for PagerDuty/Slack integration
#     },
# )
# print(f"Monitoring job: {monitoring_job.resource_name}")

print("Model monitoring configuration ready (commented for cost).")
print("\nMonitoring types:")
print("  1. Skew detection: training data vs. serving data distributions")
print("  2. Drift detection: recent serving data vs. baseline serving data")
print("  3. Attribution drift: feature importance changes over time")
print("\nThreshold meaning:")
print("  value=0.3 means alert if Jensen-Shannon divergence > 0.3")
print("  Lower thresholds = more sensitive = more alerts")

In [ ]:
# Simulate drift detection locally
from scipy.spatial.distance import jensenshannon
import numpy as np

# Training distribution of transaction amounts
train_amounts = df["transaction_amount"].values

# Simulated serving distribution (shifted — more high-value transactions)
np.random.seed(99)
serve_amounts = np.random.exponential(150, 5000)  # shifted from 100 to 150

# Compute Jensen-Shannon divergence
bins = np.linspace(0, 1000, 50)
train_hist, _ = np.histogram(train_amounts, bins=bins, density=True)
serve_hist, _ = np.histogram(serve_amounts, bins=bins, density=True)

# Add small epsilon to avoid zero bins
train_hist = train_hist + 1e-10
serve_hist = serve_hist + 1e-10

js_divergence = jensenshannon(train_hist, serve_hist)

threshold = 0.3
alert = js_divergence > threshold

print(f"Feature: transaction_amount")
print(f"  Training mean: {train_amounts.mean():.2f}")
print(f"  Serving mean:  {serve_amounts.mean():.2f}")
print(f"  JS Divergence: {js_divergence:.4f}")
print(f"  Threshold:     {threshold}")
print(f"  Alert:         {'YES - DRIFT DETECTED' if alert else 'No drift'}")

---
## 6. Endpoint Traffic Splitting Pattern

Traffic splitting enables A/B testing and canary deployments by sending different
percentages of traffic to different model versions on the same endpoint.

In [ ]:
# ============================================================
# TRAFFIC SPLITTING (commented for cost)
# ============================================================

# from google.cloud import aiplatform
#
# aiplatform.init(project=PROJECT_ID, location=REGION)
# endpoint = aiplatform.Endpoint("projects/PROJECT/locations/REGION/endpoints/ENDPOINT_ID")
#
# # Current state: model_v1 serves 100% traffic
#
# # Step 1: Deploy v2 as canary (10% traffic)
# model_v2 = aiplatform.Model("projects/PROJECT/locations/REGION/models/MODEL_V2_ID")
# model_v2.deploy(
#     endpoint=endpoint,
#     deployed_model_display_name="fraud-model-v2",
#     machine_type="n1-standard-4",
#     min_replica_count=1,
#     max_replica_count=3,
#     traffic_percentage=10,  # 10% canary
# )
# # Now: v1=90%, v2=10%
#
# # Step 2: After monitoring (24-48h), increase v2 traffic
# endpoint.update(traffic_split={
#     "deployed-model-v1-id": 50,
#     "deployed-model-v2-id": 50,
# })
# # Now: v1=50%, v2=50%
#
# # Step 3: Full rollout
# endpoint.update(traffic_split={
#     "deployed-model-v1-id": 0,
#     "deployed-model-v2-id": 100,
# })
# # Now: v2=100%
#
# # Step 4: Undeploy old model
# endpoint.undeploy(deployed_model_id="deployed-model-v1-id")

print("Traffic splitting pattern:")
print("  Phase 1: Deploy v2 at 10% (canary)")
print("  Phase 2: Monitor for 24-48h")
print("  Phase 3: Increase to 50% if metrics are stable")
print("  Phase 4: Full rollout to 100%")
print("  Phase 5: Undeploy old model")
print("\nRollback: If v2 shows issues at any phase,")
print("  set traffic_split={v1: 100, v2: 0} then undeploy v2.")

In [ ]:
# Simulate canary analysis: compare model versions
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# Prepare data
df_encoded = pd.get_dummies(df, columns=["merchant_category"])
X = df_encoded.drop(columns=["is_fraud"])
y = df_encoded["is_fraud"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Model V1: conservative hyperparameters
model_v1 = GradientBoostingClassifier(n_estimators=50, learning_rate=0.1, max_depth=3, random_state=42)
model_v1.fit(X_train, y_train)

# Model V2: aggressive hyperparameters
model_v2 = GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=5, random_state=42)
model_v2.fit(X_train, y_train)

# Compare
print(f"{'Metric':<15} {'Model V1':>10} {'Model V2':>10} {'Winner':>10}")
print("-" * 50)

for name, metric_fn in [
    ("Accuracy", lambda y_t, y_p, y_pr: accuracy_score(y_t, y_p)),
    ("F1", lambda y_t, y_p, y_pr: f1_score(y_t, y_p)),
    ("AUC-ROC", lambda y_t, y_p, y_pr: roc_auc_score(y_t, y_pr)),
]:
    v1_pred = model_v1.predict(X_test)
    v1_prob = model_v1.predict_proba(X_test)[:, 1]
    v2_pred = model_v2.predict(X_test)
    v2_prob = model_v2.predict_proba(X_test)[:, 1]

    v1_score = metric_fn(y_test, v1_pred, v1_prob)
    v2_score = metric_fn(y_test, v2_pred, v2_prob)
    winner = "V2" if v2_score > v1_score else "V1"

    print(f"{name:<15} {v1_score:>10.4f} {v2_score:>10.4f} {winner:>10}")

---
## 7. Cost Estimation for Different Deployment Configs

Understanding deployment costs is critical for the exam and real-world decisions.
Below we estimate monthly costs for various configurations.

In [ ]:
# Cost estimation calculator for Vertex AI endpoints
# Prices are approximate (us-central1, as of 2025)

PRICES = {
    # Machine type: $/hour
    "n1-standard-2": 0.095,
    "n1-standard-4": 0.190,
    "n1-standard-8": 0.380,
    "n1-standard-16": 0.760,
    "n1-highmem-2": 0.118,
    "n1-highmem-4": 0.237,
    # Accelerators: $/hour (additional)
    "NVIDIA_TESLA_T4": 0.35,
    "NVIDIA_TESLA_V100": 2.48,
    "NVIDIA_TESLA_A100": 3.67,
}

HOURS_PER_MONTH = 730  # ~30.4 days


def estimate_monthly_cost(
    machine_type: str,
    min_replicas: int,
    max_replicas: int,
    avg_utilization: float = 0.5,  # fraction of time at max
    accelerator_type: str = None,
    accelerator_count: int = 0,
):
    """Estimate monthly cost for a Vertex AI endpoint deployment."""
    machine_cost = PRICES.get(machine_type, 0)
    gpu_cost = PRICES.get(accelerator_type, 0) * accelerator_count if accelerator_type else 0
    per_replica_hour = machine_cost + gpu_cost

    # Average replicas = min + (max - min) * utilization
    avg_replicas = min_replicas + (max_replicas - min_replicas) * avg_utilization

    monthly = per_replica_hour * avg_replicas * HOURS_PER_MONTH
    return {
        "machine_type": machine_type,
        "replicas": f"{min_replicas}-{max_replicas}",
        "gpu": f"{accelerator_count}x {accelerator_type}" if accelerator_type else "None",
        "per_replica_hour": f"${per_replica_hour:.3f}",
        "avg_replicas": f"{avg_replicas:.1f}",
        "monthly_cost": f"${monthly:,.2f}",
    }


# Compare deployment configurations
configs = [
    {"machine_type": "n1-standard-2", "min_replicas": 1, "max_replicas": 1},
    {"machine_type": "n1-standard-4", "min_replicas": 1, "max_replicas": 3},
    {"machine_type": "n1-standard-4", "min_replicas": 2, "max_replicas": 5},
    {"machine_type": "n1-standard-4", "min_replicas": 1, "max_replicas": 3,
     "accelerator_type": "NVIDIA_TESLA_T4", "accelerator_count": 1},
    {"machine_type": "n1-standard-8", "min_replicas": 1, "max_replicas": 3,
     "accelerator_type": "NVIDIA_TESLA_V100", "accelerator_count": 1},
]

print(f"{'Config':<35} {'Replicas':<12} {'GPU':<20} {'$/hr/replica':<14} {'Avg Replicas':<14} {'Monthly'}")
print("=" * 115)

for i, cfg in enumerate(configs):
    result = estimate_monthly_cost(**cfg)
    label = f"{cfg['machine_type']}"
    print(f"{label:<35} {result['replicas']:<12} {result['gpu']:<20} {result['per_replica_hour']:<14} {result['avg_replicas']:<14} {result['monthly_cost']}")

In [ ]:
# Batch prediction vs Online endpoint cost comparison
print("=" * 70)
print("COST COMPARISON: Batch Prediction vs Online Endpoint")
print("=" * 70)
print()

# Scenario: Score 1M records daily
daily_records = 1_000_000
records_per_hour = 100_000  # batch throughput per replica
batch_hours = daily_records / records_per_hour
batch_replicas = 4
batch_machine_cost = 0.190  # n1-standard-4

batch_daily = batch_hours * batch_replicas * batch_machine_cost
batch_monthly = batch_daily * 30

# Online endpoint (always on)
online_monthly = 0.190 * 2 * HOURS_PER_MONTH  # 2 replicas, 24/7

print(f"Scenario: Score {daily_records:,} records per day")
print()
print(f"Batch Prediction:")
print(f"  {batch_replicas} replicas x {batch_hours:.1f} hours x ${batch_machine_cost}/hr")
print(f"  Daily: ${batch_daily:.2f}")
print(f"  Monthly: ${batch_monthly:.2f}")
print()
print(f"Online Endpoint (always-on):")
print(f"  2 replicas x 730 hours x ${batch_machine_cost}/hr")
print(f"  Monthly: ${online_monthly:.2f}")
print()
savings = ((online_monthly - batch_monthly) / online_monthly) * 100
print(f"Batch saves: {savings:.0f}% vs always-on endpoint")
print(f"\nConclusion: If you don't need real-time predictions, batch is significantly cheaper.")

---
## Summary

| Topic | Key Takeaway |
|-------|-------------|
| **AutoML** | No-code model training; use `maximize-au-prc` for imbalanced data |
| **Custom Training** | Pre-built containers for standard frameworks; custom Docker for everything else |
| **Online Endpoints** | Dedicated (always-on) or serverless; configure auto-scaling |
| **Batch Prediction** | Cost-effective for bulk offline scoring; auto-provisions compute |
| **Model Monitoring** | Skew (train vs serve), drift (over time), attribution (feature importance) |
| **Traffic Splitting** | Canary deploy at 5-10%, gradually increase, rollback if needed |
| **Cost Optimization** | Right-size machines, use batch when possible, scale-to-zero for low traffic |